# Graphene pretrained baseline

This notebook calls the separate `graphene-training` package. It does not invent
background from missing annotations, evaluate the final test split, or export ONNX.
The current lab v3 is **blocked** until eligibility/semantic review and reliable
background support exist. Local smoke tests do not establish real Colab acceptance.

Use an immutable repository revision. For a real trial, choose the account/access
method yourself, retain completed generations outside `/content`, recreate the
runtime, and prove resume. See `docs/14_COLAB_BASELINE_TRAINING.md`.


In [ ]:
from pathlib import Path
import json, subprocess, sys

REPO_URL = "https://github.com/camiloandcu/Graphene-Segmentation.git"
REPO_REVISION = "b0480172dd80181efbd7241b7c8e2853818407f4"
DEVICE = "cuda"  # Choose a GPU runtime in Colab, or explicitly select "cpu".
REPO = Path("/content/Graphene-Segmentation")
assert len(REPO_REVISION) == 40 and all(c in "0123456789abcdef" for c in REPO_REVISION), "Set the immutable code revision."
if not REPO.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", REPO_REVISION], check=True)
index = "https://download.pytorch.org/whl/cu126" if DEVICE == "cuda" else "https://download.pytorch.org/whl/cpu"
subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.7.1", "torchvision==0.22.1", "--index-url", index], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO / "packages/graphene-training/requirements-common.txt"),
    str(REPO / "packages/graphene-model-contract"), str(REPO / "packages/graphene-dataset-contract"),
    str(REPO / "packages/graphene-training")], check=True)
print("Restart the runtime after installation if any imported packages changed; then continue below.")


## Choose persistence and recover reviewed data

Set `PERSISTENCE_CHOICE` deliberately. The Drive cell uses **your manual account
selection and consent**. Download mode requires verified downloads retained outside
Colab after every completed epoch; an automatic download request is insufficient.
Copy a ready dataset folder to `DATASET` yourself and keep it unchanged across
runtimes. Do not use a raw COCO ZIP, previews, predictions, or blocked diagnostics.


In [ ]:
from pathlib import Path
import os, json, shutil
import torch
from graphene_training import TrainingConfig, preflight, run, resume
from graphene_training.storage import inspect, persist

REPO = Path("/content/Graphene-Segmentation")
os.chdir(REPO)
PERSISTENCE_CHOICE = None  # Set "drive" or "download"; choose before starting.
RUN_ID = "graphene-baseline-001"
DATASET = Path("/content/ready-dataset")
LOCAL_RUN = Path("/content/runs") / RUN_ID
assert PERSISTENCE_CHOICE in ("drive", "download"), "Choose an explicit persistence method."
if PERSISTENCE_CHOICE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")  # Execute only after choosing your account/access method.
    DURABLE_RUN = Path("/content/drive/MyDrive/Graphene/runs") / RUN_ID
else:
    DURABLE_RUN = Path("/content/download-staging") / RUN_ID  # Still transient until retained outside Colab.
assert DEVICE in ("cpu", "cuda")
assert DEVICE != "cuda" or torch.cuda.is_available(), "Select a compatible GPU runtime or explicit CPU configuration."
config = TrainingConfig(device=DEVICE, precision="float32", persistence_directory=str(DURABLE_RUN))
print(json.dumps(preflight(DATASET, config), indent=2))


## New run or fresh-runtime resume

The trainer copies and verifies each completed generation before continuing.
In download mode this notebook executes one completed epoch at a time so the
retained download can be confirmed before the next epoch. Internal stopping is
only an epoch boundary; baseline configuration and scheduler horizon stay fixed.

For a fresh runtime, reinstall the same revision/pins, restore the unchanged
reviewed dataset, and recover the retained run into `DURABLE_RUN` yourself (Drive
already exposes it). Verify `inspect`, copy to a new local folder, and set `MODE`
to `"resume"`. Corrupt/incompatible runs fail; do not edit fingerprints.


In [ ]:
MODE = "new"  # Change to "resume" only for an existing verified retained run.
if MODE == "resume":
    recovered = inspect(DURABLE_RUN)
    assert recovered["last"], "No completed retained generation."
    if not LOCAL_RUN.exists():
        LOCAL_RUN.parent.mkdir(parents=True, exist_ok=True)
        shutil.copytree(DURABLE_RUN, LOCAL_RUN)
    state = inspect(LOCAL_RUN)
    assert state["run"]["config"] == config.model_dump(), "Restore the exact saved configuration."
    next_epoch = state["completed_epochs"] + 1
    status = resume(DATASET, LOCAL_RUN, state["last"],
                    _stop_after=next_epoch if PERSISTENCE_CHOICE == "download" else None)
else:
    assert MODE == "new"
    status = run(DATASET, config, LOCAL_RUN, _stop_after=1 if PERSISTENCE_CHOICE == "download" else None)
print(json.dumps(status, indent=2))
print(json.dumps(inspect(DURABLE_RUN), indent=2))


## Retain download mode and inspect the winner

For download mode, execute the download cell, verify/retain the ZIP on your machine,
and explicitly confirm retention before continuing. To prove fresh-runtime resume,
start a new VM, reinstall, upload/extract that retained ZIP into a new folder and
verify it with `inspect` before running the resume cell. Both `/content` folders
are transient. For Drive mode the verified retained generations are already in
Drive; still recreate the runtime and resume to satisfy the real trial.


In [ ]:
if PERSISTENCE_CHOICE == "download":
    from google.colab import files
    inspect(DURABLE_RUN)
    zip_path = shutil.make_archive(str(Path("/content") / f"{RUN_ID}-completed"), "zip", DURABLE_RUN)
    files.download(zip_path)
    print("Retain and verify this ZIP outside Colab before confirming below.")
else:
    print("Verified Drive state:", inspect(DURABLE_RUN)["last"])


In [ ]:
DOWNLOAD_RETAINED = False  # Change only after retaining this epoch's verified ZIP outside Colab.
if PERSISTENCE_CHOICE == "download":
    assert DOWNLOAD_RETAINED, "A download request alone is not durable retention."
    state = inspect(LOCAL_RUN)
    if state["completed_epochs"] < state["run"]["config"]["epochs"]:
        status = resume(DATASET, LOCAL_RUN, state["last"], _stop_after=state["completed_epochs"] + 1)
        print(json.dumps(status, indent=2))
        print("Repeat download, retention confirmation, and this cell for the next epoch.")
        DOWNLOAD_RETAINED = False
winner = inspect(LOCAL_RUN)
print({k: winner[k] for k in ("last", "best", "best_score", "best_checkpoint_sha256")})


## Record real acceptance evidence

Retain run/epoch manifests and record actual reviewed dataset identity, lab review,
ImageNet checksum, configuration, Python/packages/CUDA/cuDNN/GPU, multiple
completed epochs, persistence method and retained artifact identity. Recreate the
runtime and record the resumed generation and subsequent epoch. Reconcile the
winning score and SHA-256 with development history. Unknown fractions/support
remain visible; scores measure only annotated supervision. AC-5 remains pending
until this real reviewed-data trial is documented. Final-test scoring and regional
lab-review UI belong to later work items.
